In [1]:
!pip install nlpaug

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 11.4 MB/s eta 0:00:0000:01


In [2]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


In [43]:
import pandas as pd
import random
import numpy as np
import torch
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
from sklearn.metrics import roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from torch.nn import CrossEntropyLoss
from transformers import Trainer

In [44]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [45]:
df_train = pd.read_csv('/kaggle/input/datasets/suruadelina/liar-2/train.csv', encoding="utf-8")
df_test = pd.read_csv('/kaggle/input/datasets/suruadelina/liar-2/test.csv', encoding="utf-8")
df_val = pd.read_csv('/kaggle/input/datasets/suruadelina/liar-2/valid.csv', encoding="utf-8")

In [46]:
def clean(text):
    if pd.isna(text):
        return text
    text = text.lower()
    text = text.strip()
    text = re.sub(r'https\S+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

In [47]:
label_mapping = {
    0: 0,  # Pants on Fire → Fake
    1: 0,  # False → Fake
    2: 0,  # Barely True → Fake
    3: 1,  # Half True → True
    4: 1,  # Mostly True → True
    5: 1   # True → True
}

In [48]:
df_train['label'] = df_train['label'].map(label_mapping)
df_val['label'] = df_val['label'].map(label_mapping)
df_test['label'] = df_test['label'].map(label_mapping)

print(df_train['label'].value_counts())

label
0    10591
1     7778
Name: count, dtype: int64


In [49]:
df_train['statement'] = df_train['statement'].apply(clean)
df_test['statement'] = df_test['statement'].apply(clean)
df_val['statement'] = df_val['statement'].apply(clean)

In [1]:
df_train = df_train[['statement', 'label']]
df_test = df_test[['statement', 'label']]
df_val = df_val[['statement', 'label']]

NameError: name 'df_train' is not defined

In [50]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# Tokenize without truncation
token_lengths = [len(tokenizer.tokenize(text)) for text in df_train['statement']]
max_length = max(token_lengths)

train_encodings = tokenizer(
    df_train['statement'].to_list(),
    padding=True,
    truncation=True,
    max_length=max_length
)
test_encodings = tokenizer(
    df_test['statement'].to_list(), 
    padding=True, 
    truncation=True, 
    max_length=max_length
)

val_encodings = tokenizer(
    df_val['statement'].to_list(),
    padding=True,
    truncation=True,
    max_length=max_length
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [51]:
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
        print({k: len(v) for k, v in self.encodings.items()})
        print(len(self.labels))

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [52]:
train_dataset = MyDataset(train_encodings, df_train['label'].to_list())
test_dataset = MyDataset(test_encodings, df_test['label'].to_list())
val_dataset = MyDataset(val_encodings, df_val['label'].to_list())

{'input_ids': 18369, 'token_type_ids': 18369, 'attention_mask': 18369}
18369
{'input_ids': 2296, 'token_type_ids': 2296, 'attention_mask': 2296}
2296
{'input_ids': 2297, 'token_type_ids': 2297, 'attention_mask': 2297}
2297


In [53]:
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [54]:
from sklearn.metrics import roc_auc_score, f1_score
import numpy as np
import torch

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(logits, axis=1)

    acc = (preds == labels).mean()

    f1 = f1_score(labels, preds, average="macro")

    auc = roc_auc_score(labels, probs[:, 1])  # only probability of class 1

    return {
        "accuracy": acc,
        "f1": f1,
        "auc": auc
    }

In [55]:
# train_labels = df_train['label'].to_list()

# # Get unique classes
# classes = np.unique(train_labels)

# # Compute weights
# class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_labels)

# # Convert to torch tensor
# class_weights = torch.tensor(class_weights, dtype=torch.float).to("cuda")  # or "cpu"
# print("Class weights:", class_weights)

In [56]:
# loss_fn = CrossEntropyLoss(weight=class_weights)

In [57]:
# class WeightedTrainer(Trainer):
#     def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
#         labels = inputs.get("labels")
#         outputs = model(**inputs)
#         logits = outputs.get("logits")
        
#         # Weighted cross-entropy loss
#         loss = loss_fn(logits, labels)
        
#         return (loss, outputs) if return_outputs else loss

In [58]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/output",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    learning_rate=1e-5,        
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,            
    weight_decay=0.01,
    seed=SEED,
    dataloader_num_workers=0,
    report_to='none',
    logging_strategy='steps',
    logging_steps=1,
    logging_first_step=True,
)

In [59]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [60]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

# trainer = WeightedTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset,
#     compute_metrics=compute_metrics
# )


Epoch,Training Loss,Validation Loss,Accuracy,F1,Auc
1,0.104900,0.555912,0.693513,0.691288,0.776667
2,1.583500,0.566411,0.708751,0.705427,0.783702
3,0.083300,0.639477,0.695690,0.695283,0.778554
4,0.020900,0.697487,0.679582,0.679568,0.767322
5,0.308900,0.783524,0.696125,0.693733,0.764781


TrainOutput(global_step=5745, training_loss=0.43132334816179607, metrics={'train_runtime': 999.7898, 'train_samples_per_second': 91.864, 'train_steps_per_second': 5.746, 'total_flos': 5238990764900100.0, 'train_loss': 0.43132334816179607, 'epoch': 5.0})

In [61]:
# at the end on validation data
predictions = trainer.predict(test_dataset)

final_metrics = compute_metrics(
    (predictions.predictions, predictions.label_ids)
)

print("\nFinal Evaluation:")
print(final_metrics)


Final Evaluation:
{'accuracy': np.float64(0.6907665505226481), 'f1': 0.6883059273422563, 'auc': np.float64(0.7740497592208061)}


In [63]:
trainer.save_model('/kaggle/working/BERT_model')